# 📑 Automated Reviewer Assignment & COI Flow

Welcome! This notebook provides an interactive, end-to-end pipeline to:
1. **Install dependencies & setup workspace**
2. **Upload input files (one-stop panel with dedicated buttons)**
3. **Generate Conflict of Interest (COI) matrix & report**
4. **Download COI matrix for review/editing**
5. **Upload updated COI matrix**
6. **Run Reviewer Allocation optimization (MILP)**
7. **Produce visuals, charts & download deliverables package**

---
## 🛠️ Step 1: Install Dependencies & Setup Workspace
Installs `pulp` MILP solver and optimization libraries, unzips `reviewer_assignment_colab.zip`, and sets the working directory.

In [ ]:
#@title Step 1: Install Dependencies & Setup Workspace { vertical-output: true }
import os
import sys
import zipfile
from pathlib import Path

# 1. Install required packages
!pip install -q pulp pyyaml pandas matplotlib ipywidgets

# 2. Check for uploaded zip package and unpack
zip_candidates = list(Path("/content").glob("*.zip"))
if zip_candidates:
    zip_path = str(zip_candidates[0])
    print(f"📦 Unpacking '{os.path.basename(zip_path)}' into /content...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")

# 3. Resolve workspace directory
candidates = [
    Path("/content/reviewer_assignment/coi_gen_and_reviewer_allocation"),
    Path("/content/coi_gen_and_reviewer_allocation"),
    Path.cwd() if (Path.cwd() / "coi_generation.py").exists() else None,
    Path.cwd() / "coi_gen_and_reviewer_allocation"
]
target_dir = None
for c in candidates:
    if c and c.exists() and (c / "coi_generation.py").exists():
        target_dir = c.resolve()
        break

if target_dir:
    os.chdir(str(target_dir))
    os.makedirs("input_files", exist_ok=True)
    os.makedirs("intermediate_files", exist_ok=True)
    os.makedirs("output_files", exist_ok=True)
    print(f"\n✅ Environment Ready! Current working directory: {os.getcwd()}")
else:
    print("\n⚠️ Workspace folder not found. Please drag 'reviewer_assignment_colab.zip' into Colab Files panel on the left and re-run this cell.")


---
## 📤 Step 2: Upload Input Files (All-in-One Panel)
Use the buttons below to select and upload your custom files:
- **Reviewer Expertise** (`cfg_reviewer_expertise.csv`): Reviewer roster with institutions and area scores.
- **Paper Info / Submissions** (`cfg_paper_info.csv`): Paper submissions with author affiliations.
- **Paper Technical Areas** (`cfg_paper_area.csv`): Technical area assignments for papers.
- **Must-Review Constraints** (`cfg_must_review.csv`): Pre-assigned reviewer-paper locks.
- **Existing COI Matrix** (`cfg_coi.csv`, optional): Any existing conflict matrix.

💡 *If you do not upload a file, the pre-loaded demo file is kept automatically!*

In [ ]:
#@title Step 2: Input Files Upload Panel { vertical-output: true }
import os
import ipywidgets as widgets
from IPython.display import display, HTML

os.makedirs("input_files", exist_ok=True)

# Create individual file upload buttons
btn_expertise = widgets.FileUpload(description="1. Expertise", accept=".csv", multiple=False, layout=widgets.Layout(width="220px"))
btn_info = widgets.FileUpload(description="2. Paper Info", accept=".csv", multiple=False, layout=widgets.Layout(width="220px"))
btn_areas = widgets.FileUpload(description="3. Tech Areas", accept=".csv", multiple=False, layout=widgets.Layout(width="220px"))
btn_must = widgets.FileUpload(description="4. Must-Review", accept=".csv", multiple=False, layout=widgets.Layout(width="220px"))
btn_coi = widgets.FileUpload(description="5. COI (Optional)", accept=".csv", multiple=False, layout=widgets.Layout(width="220px"))

btn_apply = widgets.Button(description="💾 Save Uploaded Files", button_style="success", icon="check", layout=widgets.Layout(width="250px", height="40px"))
output_status = widgets.Output()

def save_widget_file(widget_obj, target_path):
    if not widget_obj.value:
        return False
    if isinstance(widget_obj.value, (tuple, list)):
        item = widget_obj.value[0]
        content = item.get("content")
        with open(target_path, "wb") as f:
            f.write(bytes(content))
        return True
    elif isinstance(widget_obj.value, dict):
        first_val = next(iter(widget_obj.value.values()))
        content = first_val.get("content")
        with open(target_path, "wb") as f:
            f.write(bytes(content))
        return True
    return False

def on_apply_clicked(b):
    with output_status:
        output_status.clear_output()
        saved = []
        if save_widget_file(btn_expertise, "input_files/cfg_reviewer_expertise.csv"):
            saved.append("cfg_reviewer_expertise.csv")
        if save_widget_file(btn_info, "input_files/cfg_paper_info.csv"):
            saved.append("cfg_paper_info.csv")
        if save_widget_file(btn_areas, "input_files/cfg_paper_area.csv"):
            saved.append("cfg_paper_area.csv")
        if save_widget_file(btn_must, "input_files/cfg_must_review.csv"):
            saved.append("cfg_must_review.csv")
        if save_widget_file(btn_coi, "input_files/cfg_coi.csv"):
            saved.append("cfg_coi.csv")
        
        if saved:
            print("✅ Successfully saved uploaded files:")
            for s in saved:
                print(f"   • input_files/{s}")
        else:
            print("ℹ️ No new files selected. Existing / demo files in input_files/ will be used.")
        
        print("\n📁 Files currently staged in input_files/:")
        for f in sorted(os.listdir("input_files")):
            size_kb = os.path.getsize(os.path.join("input_files", f)) / 1024
            print(f"   • {f:<28} ({size_kb:.1f} KB)")

btn_apply.on_click(on_apply_clicked)

print("Choose files for any items you wish to update, then click 'Save Uploaded Files':")
display(widgets.VBox([
    widgets.HBox([widgets.Label("Reviewer Expertise:", layout=widgets.Layout(width="170px")), btn_expertise]),
    widgets.HBox([widgets.Label("Paper Info / Submissions:", layout=widgets.Layout(width="170px")), btn_info]),
    widgets.HBox([widgets.Label("Paper Technical Areas:", layout=widgets.Layout(width="170px")), btn_areas]),
    widgets.HBox([widgets.Label("Must-Review Constraints:", layout=widgets.Layout(width="170px")), btn_must]),
    widgets.HBox([widgets.Label("Existing COI (Optional):", layout=widgets.Layout(width="170px")), btn_coi]),
    widgets.HTML("<br>"),
    btn_apply,
    output_status
]))


---
## 🔍 Step 3: Run COI Generation
Extracts author affiliations from `cfg_paper_info.csv` and matches them against reviewer affiliations from `cfg_reviewer_expertise.csv` to generate the Conflict of Interest (COI) matrix and audit report.

In [ ]:
#@title Step 3: Generate COI Matrix & Audit Report { vertical-output: true }
import os

!python3 coi_generation.py

report_path = "output_files/coi_report.txt"
if os.path.exists(report_path):
    print("\n" + "="*50)
    print("📋 CONFLICT OF INTEREST REPORT PREVIEW:")
    print("="*50)
    with open(report_path, "r") as f:
        print(f.read())


---
## ⬇️ Step 4: Download COI Matrix for Review
Downloads the newly generated `cfg_coi.csv`. You can open this CSV file in Excel or Google Sheets to inspect the detected conflicts and make manual additions or removals (1 for conflict, 0 for clear).

In [ ]:
#@title Step 4: Download Generated COI Matrix
from google.colab import files
import os

coi_file = "input_files/cfg_coi.csv"
if os.path.exists(coi_file):
    print(f"⬇️ Downloading '{coi_file}' to your browser...")
    files.download(coi_file)
else:
    print(f"Error: '{coi_file}' not found. Please run Step 3 first.")


---
## ⬆️ Step 5: Upload Modified COI Matrix
Upload your revised `cfg_coi.csv`. This will overwrite `input_files/cfg_coi.csv` and be used in the optimization solver in Step 6.
*(If you made no manual changes, you can simply skip this step and proceed to Step 6)*.

In [ ]:
#@title Step 5: Upload Modified COI Matrix (Optional)
from google.colab import files
import os

print("📤 Select your edited 'cfg_coi.csv' file to upload (or skip if no edits):")
uploaded = files.upload()

if uploaded:
    for fname in uploaded.keys():
        os.replace(fname, "input_files/cfg_coi.csv")
        print(f"✅ Successfully staged '{fname}' as 'input_files/cfg_coi.csv'.")
else:
    print("ℹ️ No file uploaded. Continuing with autogenerated COI matrix.")


---
## 🚀 Step 6: Run Reviewer Allocation (MILP Optimizer)
Runs the Mixed-Integer Linear Programming optimization to assign reviewers, appoint balanced Champions and Co-Champions (excluding Chair/Co-Chair), and maximize matching expertise.

In [ ]:
#@title Step 6: Execute Reviewer Allocation Flow { vertical-output: true }
!python3 reviewer_allocation.py


---
## 📊 Step 7: View Visualizations & Download Deliverables
Displays all workload distribution charts inline and downloads the final `reviewer_assignment_results.zip` package containing all assignment matrices, vectors, and reports.

In [ ]:
#@title Step 7: Display Charts & Download Complete Deliverables Package { vertical-output: true }
import glob
import os
from IPython.display import Image, display
from google.colab import files

print("📊 Workload & Assignment Distributions:\n")
chart_images = sorted(glob.glob("output_files/*.png"))
if chart_images:
    for img_path in chart_images:
        print(f"\n📈 {os.path.basename(img_path)}")
        display(Image(img_path, width=700))
else:
    print("No charts found in output_files/.")

!zip -r -q /content/reviewer_assignment_results.zip output_files/
print("\n⬇️ Downloading deliverables package: reviewer_assignment_results.zip")
files.download("/content/reviewer_assignment_results.zip")
